In [ ]:
%pip install langchain langchain-community langchainhub langchain-classic beautifulsoup4 yfinance gradio

%pip install -qU langchain-openai langchain-text-splitters langchain-chroma

In [1]:
import os
import pprint
import getpass
import bs4
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from google.colab import userdata

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

/tmp/ipykernel_26562/2679565371.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


==============================================
## Initializing LLM Client
==============================================

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

==============================================
## Setup and Data-loading
==============================================

In [3]:
def get_sitemap(url):
    req = Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response = urlopen(req)
    xml = BeautifulSoup(
        response,
        "lxml-xml",
        from_encoding=response.info().get_param("charset")
    )
    return xml

def get_urls(xml, name=None, data=None, verbose=False):
    urls = []
    for url in xml.find_all("url"):
        loc = url.find("loc")
        if loc:
            urls.append(loc.get_text(strip=True))
    return urls

url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
xml = get_sitemap(url)
urls = get_urls(xml, verbose=False)

docs = []
for i, url in enumerate(urls[:114], start=1):
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
    print("Loaded document index:", i)

Loaded document index: 1
Loaded document index: 2
Loaded document index: 3
Loaded document index: 4
Loaded document index: 5
Loaded document index: 6
Loaded document index: 7
Loaded document index: 8
Loaded document index: 9
Loaded document index: 10
Loaded document index: 11
Loaded document index: 12
Loaded document index: 13
Loaded document index: 14
Loaded document index: 15
Loaded document index: 16
Loaded document index: 17
Loaded document index: 18
Loaded document index: 19
Loaded document index: 20
Loaded document index: 21
Loaded document index: 22
Loaded document index: 23
Loaded document index: 24
Loaded document index: 25
Loaded document index: 26
Loaded document index: 27
Loaded document index: 28
Loaded document index: 29
Loaded document index: 30
Loaded document index: 31
Loaded document index: 32
Loaded document index: 33
Loaded document index: 34
Loaded document index: 35
Loaded document index: 36
Loaded document index: 37
Loaded document index: 38
Loaded document index

==============================================
## Vector-store and Retriever
==============================================

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

print(f"Len docs: {len(docs)}, Len splits: {len(splits)}")

Len docs: 114, Len splits: 2714


 ==============================================
## RAG Chain Setup
==============================================

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = (
    "You are a financial assistant for question-answering tasks."
    "Use ONLY the following pieces of retrieved context to answer the question."
    "If you don't know the answer, say that you don't know."
    "Use three sentences maximum and keep the answer concise."
    "CRITICAL: Only use information from the context below."
    "If you cannot find the answer in the context, respond with: "
    "I cannot find that information in the available documents."
    "If the question is not clear ask follow up questions \n\n"
    "{context}"
)

# Retriever History
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

==============================================
## Run Gradio UI
==============================================

In [6]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage

def content_to_text(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        )
    return str(content) if content is not None else ""

def predict(message, history):
    history_for_llm = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content", "")
        else:
            role = getattr(item, "role", None)
            content = getattr(item, "content", "")
        content = content_to_text(content)

        if role == "user":
            history_for_llm.append(HumanMessage(content=content))
        elif role == "assistant":
            history_for_llm.append(AIMessage(content=content))
    result = rag_chain.invoke(
        {
            "input": message,
            "chat_history": history_for_llm,
        }
    )
    return result["answer"]


def undo_last(history):
    """Remove the latest user/assistant turn."""
    if not history:
        return []
    return history[:-2] if len(history) >= 2 else []


with gr.Blocks() as demo:
    gr.Markdown("# Finance_Pulse_QnA_BoT")
    chatbot = gr.Chatbot(
        height=400,
        allow_tags=False,
    )
    msg = gr.Textbox(
        placeholder=(
            "Hi! I am your virtual assistant, "
            "how can I help you today?"
        ),
        container=False,
        scale=7,
    )
    with gr.Row():
        undo = gr.Button("Delete Previous")
        clear = gr.Button("Clear")
    gr.ChatInterface(
        fn=predict,
        chatbot=chatbot,
        textbox=msg,
        examples=[
            "What is an index fund?",
            "Where can I buy stocks?",
        ],
    )
    clear.click(
        fn=lambda: [],
        inputs=None,
        outputs=chatbot,
    )
    undo.click(
        fn=undo_last,
        inputs=chatbot,
        outputs=chatbot,
    )

demo.launch(
    share=True,
    debug=False,
    theme=gr.themes.Soft(),
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c40403b31663035157.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
